# Oblsk Negotiation Agent — full walkthrough

An agent that prices every creator deal and negotiates like a real person, with a human approving each reply until autonomy is earned.

Two halves meet at one handoff:

- **the calculator** (`ev_engine` + `pricing`) fits a heavy-tailed model of the creator's views, runs a Monte Carlo, and derives the pricing ladder — **anchor** (where we open), **target** (the fee that hits our ROI goal), **walk-away** (the ceiling the downside still justifies);
- **the negotiator** (`behavior_tree` + `prose` + `qa`) runs the conversation: one pass per message, first branch that fits wins, every dollar read off the ladder.

An LLM (Claude) writes the outgoing words and reads the incoming ones; it never invents a price or picks the move. Every section below runs offline too — templates and keyword rules stand in when no key is set.

This notebook walks every feature: fitting, the ladder, authenticity risk, campaign config, the negotiation ladder end to end, human-in-the-loop moves, multi-format bundles, real-thread replay, the event log, batch metrics, and sparring against Claude.

In [ ]:
# Setup — works in Colab (clones the repo) and locally (finds it).
import importlib.util, os, subprocess, sys

if not os.path.isdir('examples') and os.path.isdir('../examples'):
    os.chdir('..')          # local Jupyter: this notebook lives in notebooks/
if importlib.util.find_spec('oblsk_negotiator') is None and not os.path.isdir('oblsk_negotiator'):
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/Sophie-S-Z/oblsk-negotiator.git'], check=True)
    os.chdir('oblsk-negotiator')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'numpy', 'scipy', 'pyyaml', 'matplotlib', 'anthropic'], check=True)
print('working dir:', os.getcwd())

In [ ]:
# Optional: enable the LLM. In Colab, store ANTHROPIC_API_KEY under the key
# icon (Secrets) and this cell picks it up. Skip it to run fully offline.
import os
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    pass
if os.environ.get('ANTHROPIC_API_KEY'):
    os.environ.pop('OBLSK_NO_LLM', None)
    print('LLM enabled: Claude drafts the messages and reads incoming text')
else:
    os.environ['OBLSK_NO_LLM'] = '1'
    print('running offline: template messages, keyword reading')

## 1. The calculator

### 1.1 A creator's views are heavy-tailed

Most posts land near the median; a few run far past it. The calculator fits a log-normal body and splices on a Pareto tail when the history demands one, then simulates thousands of outcomes. The ladder is derived from that distribution: the p10 downside sets the walk-away, the (whale-capped) expectation sets the target.

In [ ]:
import numpy as np
from oblsk_negotiator import (CreatorEconomics, fit_view_model,
                              price_ladder, PricingPolicy)

rng = np.random.default_rng(7)
views = np.concatenate([rng.lognormal(np.log(55000), 0.6, 24), [400000, 900000]])
vm = fit_view_model(views)
econ = CreatorEconomics(conversion_rate=0.0016, ltv_usd=80)

ladder = price_ladder(vm, econ, PricingPolicy())
print(vm.notes)
print(ladder.summary())

In [ ]:
import matplotlib.pyplot as plt

samples = vm.sample(20000, np.random.default_rng(1)) * econ.revenue_per_view
plt.figure(figsize=(9, 4))
plt.hist(np.clip(samples, 0, np.quantile(samples, 0.99)), bins=80, alpha=0.7)
for x, label in [(ladder.anchor, 'anchor'), (ladder.target, 'target'),
                 (ladder.walk_away, 'walk-away')]:
    plt.axvline(x, ls='--', lw=1.5, label=f'{label} ${x:,.0f}')
plt.legend(); plt.xlabel('revenue per video ($)')
plt.title('One video, simulated — and the ladder it implies')
plt.show()

### 1.2 Recency weighting and the sponsored haircut

Old posts shouldn't drag down a rising creator: pass `timestamps` and the fit weights each post by a 90-day half-life. And sponsored posts under-deliver organic history, so `sponsored_factor=0.8` scales the whole model before pricing.

In [ ]:
now = 1_780_000_000
hist = [20000] * 12 + [80000] * 12           # old posts at 20k, recent at 80k
ts = [now - 400 * 86400] * 12 + [now - 5 * 86400] * 12

flat_fit = fit_view_model(hist)
recency_fit = fit_view_model(hist, timestamps=ts)
sponsored_fit = fit_view_model(hist, timestamps=ts, sponsored_factor=0.8)
print(f'unweighted median:        {flat_fit.median_views:>10,.0f} views')
print(f'recency-weighted median:  {recency_fit.median_views:>10,.0f} views')
print(f'+ sponsored haircut:      {sponsored_fit.median_views:>10,.0f} views')

### 1.3 Authenticity: inflated accounts get a tighter ceiling

Three transparent checks (engagement per follower vs. tier norm, comments-to-likes, views-to-followers) score an account 0.2–1.0. The score discounts the downside, and since the walk-away is built from the downside, weak authenticity automatically tightens how far the agent will ever go.

In [ ]:
from oblsk_negotiator import authenticity

healthy = authenticity([{'views': 50000, 'likes': 4000, 'comments': 90}] * 10,
                       followers=100_000)
inflated = authenticity([{'views': 1500, 'likes': 300, 'comments': 0}] * 10,
                        followers=900_000)
for name, a in [('healthy', healthy), ('inflated', inflated)]:
    lad = price_ladder(vm, econ, PricingPolicy(risk_discount=a['score']))
    print(f"{name}: score {a['score']:.2f}  ->  walk-away ${lad.walk_away:,.0f}")
    for flag in a['flags']:
        print('   -', flag)

## 2. One YAML file per campaign

Everything campaign-specific lives in one file: the brief the agent answers questions from, the economics that price every deal, the negotiation stance, call windows, and how to recognize our side of an email thread. `examples/unest_campaign.yaml` was built from UNest's real program agreement and content guide.

In [ ]:
from oblsk_negotiator import load_campaign

camp = load_campaign('examples/unest_campaign.yaml')
print('campaign:   ', camp.name)
print('brand:      ', camp.brief.brand)
print('deliverables:', camp.brief.deliverables)
print('economics:  ', camp.econ)
print(f'stance:      roi_target {camp.ctx.roi_target}x | anchor_factor '
      f'{camp.ctx.anchor_factor} | approval ceiling ${camp.ctx.auto_send_dollar_ceiling:,.0f}')
print('call windows:', camp.ctx.call_windows)

# The same creator priced under UNest's economics:
vm_creator = fit_view_model(np.random.default_rng(2).lognormal(np.log(150000), 0.6, 24))
print()
print('ladder for a ~150k-view creator under this campaign:')
print(price_ladder(vm_creator, camp.econ, camp.ctx.pricing_policy()).summary())

## 3. A negotiation, end to end

A rule-based simulated creator plays the other side (their floor and temperament are constructor parameters). Watch the approval gate: every outbound message is proposed, reviewed, then sent — and each move prints its rationale.

In [ ]:
from oblsk_negotiator import (CampaignBrief, CampaignContext, SimCreator,
                              run_negotiation)

brief = CampaignBrief(brand='Aurora Skincare', product='the daily SPF serum')
creator = SimCreator(reservation_per_video=2450, opens_with='question',
                     questions=['What would the deliverables be?',
                                'And the timeline?'],
                     counter_ratio=0.6, bulk_tolerance=0.06, rng_seed=3)
outcome = run_negotiation(vm, econ, CampaignContext(), creator,
                          brief=brief, creator_name='Maya', verbose=True)
print()
print('status:', outcome.status, '| final:', outcome.final_total,
      '| rounds:', outcome.rounds, '| ROI:',
      f'{outcome.final_ev_roi:.1f}x' if outcome.final_ev_roi else '-')

### 3.1 The full ladder: revise once, restructure, escalate

A creator whose floor sits above our target forces every rung: open at the anchor, revise once to the target (never the ceiling), reshape into a bundle — more videos at a held rate, each one an independent draw at the tail — and hand a stalled thread to a person.

In [ ]:
hard = SimCreator(reservation_per_video=4200, counter_ratio=0.2,
                  bulk_tolerance=0.0, rng_seed=0)
outcome = run_negotiation(vm, econ, CampaignContext(), hard,
                          creator_name='Jordan', verbose=True)
print()
print('status:', outcome.status)

### 3.2 The human gate, both directions

A reviewer can reject any proposed message — the thread escalates instead of sending. And in autonomous mode the agent sends on its own, but anything above the campaign's dollar ceiling still requires sign-off.

In [ ]:
from oblsk_negotiator import AutonomyLevel, reject_action
from oblsk_negotiator.behavior_tree import Action

vetoed = run_negotiation(vm, econ, CampaignContext(),
                         SimCreator(reservation_per_video=2450, rng_seed=1),
                         approver=reject_action(Action.OPENING_FLAT))
print('reviewer rejected the opener ->', vetoed.status)

auto = run_negotiation(vm, econ, CampaignContext(),
                       SimCreator(reservation_per_video=1700, counter_ratio=0.7,
                                  bulk_tolerance=0.08, rng_seed=1),
                       autonomy=AutonomyLevel.AUTONOMOUS)
print('autonomous close  ->', auto.status,
      f'at ${auto.final_total:,.0f} with {auto.approvals_requested} approvals')

## 4. Multi-format deals from a pasted rate card

Creators sell formats, not generic videos. Paste their rate card as text; the agent parses it, keeps only the formats whose reach justifies the price, applies the market-standard combo discount, and offers a package that reads as a normal deal to them and clears ROI for us.

In [ ]:
from oblsk_negotiator.rate_card import parse_rate_card, IG_REEL, IG_STORY, TIKTOK
from oblsk_negotiator.ev_engine import FormatCatalog, FormatSpec

card_text = '1x Instagram Reel: $16,500' + chr(10)
card_text += '1x IG Story: $5,850' + chr(10)
card_text += '1x TikTok video: $16,750' + chr(10)
card_text += 'IG/TikTok syndication combo: $28,000'
card = parse_rate_card(card_text, creator='Karissa')
print('parsed rate card:', card.prices)   # combo line skipped on purpose

econ_mf = CreatorEconomics(conversion_rate=0.001, ltv_usd=50)
def reach(median, seed):
    return fit_view_model(np.random.default_rng(seed).lognormal(np.log(median), 0.6, 24))
catalog = FormatCatalog({
    IG_REEL:  FormatSpec(IG_REEL,  reach(1_200_000, 1), econ_mf),
    IG_STORY: FormatSpec(IG_STORY, reach(800_000, 2),  econ_mf),
    TIKTOK:   FormatSpec(TIKTOK,   reach(1_500_000, 3), econ_mf),
})
brief_mf = CampaignBrief(primary_format=IG_REEL,
                         allowed_formats=[IG_REEL, IG_STORY, TIKTOK])

manager = SimCreator(reservation_per_video=48000, counter_ratio=0.2,
                     rate_card=card, max_bundle_discount=0.20, rng_seed=0)
outcome = run_negotiation(reach(1_200_000, 1), econ_mf, CampaignContext(),
                          manager, brief=brief_mf, catalog=catalog,
                          rate_card=card, creator_name='Karissa', verbose=True)
print()
print('status:', outcome.status, '| final:', outcome.final_total)

## 5. Reading real messages

Real threads don't arrive as tidy intents. The interpretation layer reads each creator message into an intent, a dollar ask (normalized per video), and flags the tree acts on. With a key, Claude does this via structured output; offline, keyword rules approximate it.

In [ ]:
from oblsk_negotiator import interpret_message

for text in [
    "I'm usually at $3k but could you do $2,500?",
    'I charge $1,200 per video for 3 videos.',
    "That works for me, let's do it!",
    'Would love to hop on a quick call this week to discuss.',
    "As discussed on our call, we're aligned on next steps.",
    "Section 9's exclusivity provision needs $2,000 per deliverable.",
]:
    m = interpret_message(text)
    flags = [f for f, on in [('contract', m.contract_terms),
                             ('wants_call', m.wants_call),
                             ('refers_to_call', m.refers_to_call)] if on]
    ask = f' ask ${m.ask_total_usd:,.0f}/{m.ask_video_count or 1}v' if m.ask_total_usd else ''
    print(f"{m.intent.value:12s}{ask:20s} {'+'.join(flags):24s} <- {text[:58]}")

## 6. Replay a real thread

The practical test: shadow-run the agent over an email thread the team handled manually — a raw Gmail paste, signatures, quoted replies and all. At each creator message the report shows how the agent read it, the move it would have made, the message it would have sent, and what the team actually sent. Nothing is ever sent.

`examples/unest_thread.txt` is a real agency-run negotiation. Watch the human-in-the-loop moves:

- **hold_firm** — a follow-up with no counter restates the standing offer, never an unforced concession;
- **propose_call** — when they want a call, the agent proposes the campaign's call windows and flags a teammate to run it (`NEEDS A HUMAN`);
- **escalate_human** — the contract-revision email (exclusivity, equity, kill fee) hands the thread to a person: the agent never negotiates paper.

In [ ]:
from oblsk_negotiator import replay_thread

thread = open('examples/unest_thread.txt', encoding='utf-8').read()
report = replay_thread(thread, vm_creator, camp.econ, camp.ctx,
                       brief=camp.brief, us_aliases=camp.us_aliases,
                       creator_name='Karissa')
print(report.report())

### 6.1 Asking instead of bluffing

When the agent is missing context only the team has — a question outside the brief, or terms agreed on a call it wasn't part of — it pauses the thread and asks for exactly what it needs:

In [ ]:
from oblsk_negotiator import NegotiationState, decide
from oblsk_negotiator.replay import heuristic_interpret

state = NegotiationState('c', 'unest', 't1')
msg = heuristic_interpret('Is UNest FDIC insured? My followers will ask.')
d = decide(msg, state, vm_creator, camp.econ, camp.ctx, brief=camp.brief)
print('action:', d.action.value)
print('asks the team:', d.human_prompt)

## 7. The event log

Every negotiation is also written as an append-only event log (message received, decision, approval, sent). Folding the log reproduces the negotiation's facts exactly — the audit trail is the source of truth, not a side effect.

In [ ]:
from oblsk_negotiator.events import fold_events

o = run_negotiation(vm, econ, CampaignContext(),
                    SimCreator(reservation_per_video=2450, rng_seed=5))
folded = fold_events(o.event_log)
print('runner said: ', o.status, o.rounds, 'rounds, final', o.final_total)
print('log folds to:', folded['status'], folded['round_count'],
      'rounds, final', folded['final_total'])
assert folded['status'] == o.status and folded['final_total'] == o.final_total

## 8. Metrics across many creators

In [ ]:
from oblsk_negotiator import batch_metrics

outcomes = []
for seed in range(20):
    r = np.random.default_rng(seed)
    v = fit_view_model(r.lognormal(np.log(r.uniform(35000, 75000)), 0.62, 24))
    target = price_ladder(v, econ, PricingPolicy()).target
    c = SimCreator(reservation_per_video=target * float(r.uniform(0.6, 1.4)),
                   counter_ratio=float(r.uniform(0.3, 0.75)),
                   bulk_tolerance=float(r.uniform(0, 0.12)), rng_seed=seed)
    outcomes.append(run_negotiation(v, econ, CampaignContext(), c))
print(batch_metrics(outcomes).report())

## 9. Spar against Claude

Beyond history and rule-based sims: Claude role-plays the creator's talent manager with a private playbook you write, and the agent negotiates back through its real pipeline. Judge the transcript — did it read each message right, concede only when it should, pull in a person at the right moments? Requires the API key from the setup cell.

In [ ]:
from oblsk_negotiator import spar
from oblsk_negotiator.llm import llm_available

if llm_available():
    result = spar(vm, econ, CampaignContext(), brief=brief,
                  counterpart_profile=('the creator floor is $2,600/video; '
                                       'open by asking $4,500; cite her reach; '
                                       'concede slowly; suggest a call once'),
                  creator_name='Maya')
    print(result.report())
else:
    print('Set ANTHROPIC_API_KEY in the setup cell to spar against a '
          'Claude-played manager.')

## Where to go next

- **New campaign:** copy `examples/unest_campaign.yaml`, fill in the brief and economics, tune the stance. That file is the whole per-client surface.
- **Terminal:** `py demo.py --campaign ... --replay your_thread.txt` runs section 6 from the shell; `--spar` runs section 9; `--count 20 --quiet` runs section 8.
- **Calibrate before trusting absolute dollars:** the funnel economics and the CPM table are placeholders — `docs/CALCULATOR.md` explains every number and how to defend it to a client.
- **User guide:** `docs/AGENT_GUIDE.pdf` — three pages, covers everything above.